# Direction model: will the next 15-minute candle close higher?

**Owner:** Dennis · **Stage 2** · task `direction`, horizon 1 candle, interval 15m

This notebook trains the model behind the API endpoint `/predict/{symbol}`. It uses the gold layer:

- `gold_features`: 32 features per closed candle, each built only from candles up to that one (no look-ahead)
- `outcomes`: what really happened after the candle. `realized_dir = 1` means the next candle closed higher

**Steps:** 1. load the data · 2. split it by time · 3. compare a baseline and two models on validation ·
4. test the chosen model once · 5. look at what it learned · 6. save, register and activate it · 7. call `predict()`

The logic lives in `src/models/train_model.py`, so the same code can later run as a scheduled job
(`python src/models/train_model.py --activate`). This notebook calls those functions step by step and explains the results.

**Before running:** Docker is up, and `extract_rest.py` → `transform.py` → `src/features/build_features.py` have run.
Kernel: the project's `.venv` (`pip install -r requirements.txt jupyter`).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

# work from the repo root, whether the notebook is started from the root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "sql").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src" / "models"))

import train_model as tm          # noqa: E402
from db import get_pg_connection  # noqa: E402  (train_model puts src/data on the path)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.width", 140)
print("features used:", len(tm.FEATURE_COLS), "| feature set:", tm.FEATURE_SET,
      "| target:", tm.TARGET, "| horizon:", tm.HORIZON, "candle(s)")

features used: 32 | feature set: v1 | target: realized_dir | horizon: 1 candle(s)


## 1. Load the data

One row per closed candle: the features at candle *t* plus the outcome of candle *t + 1*.
The first ~96 candles of each symbol are dropped because their 1-day features need 96 earlier candles.

In [2]:
conn = get_pg_connection()
with conn.cursor() as cur:
    cur.execute("SET TIME ZONE 'UTC'")   # show every timestamp from the database in UTC
conn.commit()

data = tm.load_symbols(conn, tm.SYMBOLS)

overview = data.groupby("symbol").agg(
    rows=("open_time", "size"), first=("open_time", "min"), last=("open_time", "max"),
    up_share=(tm.TARGET, "mean"))
overview

,rows,first,last,up_share
symbol,,,,
BTCUSDT,17183,2026-05-22 12:00:00+00:00,2026-11-17 11:30:00+00:00,0.4907
SOLUSDT,17183,2026-05-22 12:00:00+00:00,2026-11-17 11:30:00+00:00,0.4860


**How to read this:** `up_share` is the share of candles followed by a higher close. It is close to 50 %,
so a model that always says "up" (or always "down") is right about half the time. That is the bar to beat.

In [3]:
# How big is a typical 15-minute move compared with trading costs?
# Binance spot charges ~0.1 % per trade, so a round trip (buy + sell) costs ~0.2 %.
move = data.groupby("symbol")["realized_ret"].agg(lambda r: r.abs().mean() * 100).rename("avg |move| %")
move.to_frame().assign(**{"round-trip fee %": 0.2})

,avg |move| %,round-trip fee %
symbol,,
BTCUSDT,0.1337,0.2000
SOLUSDT,0.2130,0.2000


If the average move is not much larger than the fee, even a model that is right more often than not
would struggle to make money. Keep that in mind when reading the results: this project is about the pipeline,
and the evaluation has to be honest.

## 2. Split by time

Prices are a time series, so the split is chronological, never shuffled: **70 % train, 15 % validation, 15 % test**.
Shuffling would let the model learn from days that come *after* the days it is tested on.

- The cut-off dates are the same for BTC and SOL. SOL's features contain BTC returns, so cutting them on different
  dates would leak test-period information through the other coin.
- **Purge:** the label of candle *t* depends on candle *t + 1*. The last training candle before the cut is left
  out, so no training label uses a price from the validation period. The same applies before the test cut.

In [4]:
val_start, test_start = tm.time_cutoffs(data)
print(f"validation from {val_start:%Y-%m-%d %H:%M} UTC, test from {test_start:%Y-%m-%d %H:%M} UTC")

rows = []
for symbol in tm.SYMBOLS:
    parts = tm.split_by_time(data[data["symbol"] == symbol], val_start, test_start)
    for name, part in zip(("train", "validation", "test"), parts):
        rows.append({"symbol": symbol, "part": name, "rows": len(part),
                     "from": part["open_time"].min(), "to": part["open_time"].max(), "up_share": part[tm.TARGET].mean()})
pd.DataFrame(rows).set_index(["symbol", "part"])

validation from 2026-09-24 19:00 UTC, test from 2026-10-21 15:15 UTC


rows                      from                        to  up_share
symbol  part                                                                           
BTCUSDT train       12027 2026-05-22 12:00:00+00:00 2026-09-24 18:30:00+00:00    0.4920
        validation   2576 2026-09-24 19:00:00+00:00 2026-10-21 14:45:00+00:00    0.5058
        test         2578 2026-10-21 15:15:00+00:00 2026-11-17 11:30:00+00:00    0.4694
SOLUSDT train       12027 2026-05-22 12:00:00+00:00 2026-09-24 18:30:00+00:00    0.4868
        validation   2576 2026-09-24 19:00:00+00:00 2026-10-21 14:45:00+00:00    0.4938
        test         2578 2026-10-21 15:15:00+00:00 2026-11-17 11:30:00+00:00    0.4740

## 3. Compare models on validation

Three candidates, trained on *train* and judged on *validation* only. The test part stays untouched:

| Name | What it is |
|---|---|
| `baseline` | No model: always predicts the share of up candles seen in training. AUC 0.5 by definition |
| `logreg` | Logistic regression on standardised features. Simple, fast, easy to explain |
| `hgb` | Gradient-boosted trees (`HistGradientBoostingClassifier`). Can find non-linear patterns; kept small so it cannot memorise noise |

**Metrics:**
- `auc`: how well up-candles are ranked above down-candles. 0.5 = coin flip, 1.0 = perfect
- `accuracy`: share of correct up/down calls at 50 %
- `log_loss`: quality of the probabilities; ~0.693 is what "always 50 %" gets, lower is better

Besides one model per symbol, we also build one **shared** model for both symbols (row `ALL`). The features are
relative (returns, ratios), so BTC and SOL rows are comparable, and the shared model gets twice the data.

In [5]:
targets = list(tm.SYMBOLS) + [None]          # None = shared model for all symbols
built = {t: tm.build(data, t, val_start, test_start) for t in targets}

comparison = pd.concat({(t or "ALL"): b["comparison"] for t, b in built.items()}, names=["symbol", "model"])
comparison[["val_n", "val_auc", "val_accuracy", "val_log_loss"]]

val_n  val_auc  val_accuracy  val_log_loss
symbol  model                                               
BTCUSDT baseline   2576   0.5000        0.4942        0.6935
        logreg     2576   0.5273        0.5233        0.6933
        hgb        2576   0.5451        0.5330        0.6909
SOLUSDT baseline   2576   0.5000        0.5062        0.6932
        logreg     2576   0.5232        0.5198        0.6934
        hgb        2576   0.5283        0.5159        0.6947
ALL     baseline   5152   0.5000        0.5002        0.6934
        logreg     5152   0.5241        0.5182        0.6935
        hgb        5152   0.5328        0.5210        0.6929

In [6]:
picked = pd.DataFrame({(t or "ALL"): {"picked": b["name"], "val_auc": b["comparison"].loc[b["name"], "val_auc"]}
                       for t, b in built.items()}).T
picked

,picked,val_auc
BTCUSDT,hgb,0.5451
SOLUSDT,hgb,0.5283
ALL,hgb,0.5328


The model with the highest validation AUC is picked per row. Validation decides, never test: choosing by the
test score would make the test score optimistic.

## 4. Test the chosen models (once)

The picked model is refit on **train + validation** (more, and more recent, data) and scored once on the test part.
For comparison, the baseline accuracy is what "always predict the more common direction" would have scored on the same rows.

In [7]:
results = pd.DataFrame([tm.summary(b) for b in built.values()]).set_index("symbol")
results

,model,val_auc,test_auc,test_accuracy,baseline_accuracy,n_test
symbol,,,,,,
BTCUSDT,hgb,0.5451,0.5100,0.5019,0.5306,2578
SOLUSDT,hgb,0.5283,0.5103,0.5101,0.5260,2578
ALL,hgb,0.5328,0.5169,0.5116,0.5283,5156


In [8]:
# the shared model, scored separately on each coin's test rows
shared = built[None]["metrics"]
pd.Series({s: shared.get(f"test_auc_{s}") for s in tm.SYMBOLS}, name="shared model test AUC").to_frame()

,shared model test AUC
BTCUSDT,0.5135
SOLUSDT,0.5205


**How to read this** (write the real numbers into the report):
- A test AUC around **0.50–0.53** is typical for the direction of the next 15 minutes. Short-term prices are close to random.
- Accuracy only a little above the baseline accuracy means the model adds little on top of guessing the more common direction.
- If the per-symbol model beats the shared one on its coin's test AUC, coin-specific behaviour matters more than extra data,
  and the other way round.
- An honest "barely better than a coin flip, and below trading costs" is a valid result. Claiming more would need
  far stronger evidence (several test periods, fees included).

## 5. What did the models learn?

For logistic regression the coefficients show the direction of each feature's influence (features are standardised,
so sizes are comparable). For the tree model, permutation importance measures how much the validation AUC drops
when one feature is shuffled.

In [9]:
from sklearn.inspection import permutation_importance


def top_features(b, n=10):
    model = b["model"]
    if b["name"] == "logreg":
        coef = pd.Series(model[-1].coef_[0], index=tm.FEATURE_COLS)
        return coef.reindex(coef.abs().sort_values(ascending=False).index).head(n).rename("coefficient")
    part = data if b["symbol"] is None else data[data["symbol"] == b["symbol"]]
    _, val, _ = tm.split_by_time(part, val_start, test_start)
    imp = permutation_importance(model, val[tm.FEATURE_COLS], val[tm.TARGET], scoring="roc_auc",
                                 n_repeats=3, random_state=0)
    return pd.Series(imp.importances_mean, index=tm.FEATURE_COLS).sort_values(ascending=False).head(n).rename("AUC drop")


pd.concat({(t or "ALL"): top_features(b).rename_axis("feature").reset_index() for t, b in built.items()}, axis=1)

BTCUSDT                  SOLUSDT                      ALL         
          feature AUC drop         feature AUC drop         feature AUC drop
0           ret_1   0.0214          ret_96   0.0286  lower_wick_rel   0.0134
1      ret_1_lag1   0.0174    vol_ratio_16   0.0187      ret_1_lag1   0.0123
2  lower_wick_rel   0.0169          ret_16   0.0135          ret_96   0.0122
3    hl_range_rel   0.0156  upper_wick_rel   0.0124        hour_cos   0.0102
4        hour_cos   0.0146    ema_26_ratio   0.0118       btc_ret_1   0.0099
5      ret_1_lag2   0.0103  lower_wick_rel   0.0104    sma_96_ratio   0.0093
6    vol_ratio_16   0.0099           rv_96   0.0100          ret_16   0.0093
7        macd_rel   0.0097    sma_20_ratio   0.0092      ret_1_lag3   0.0080
8           rv_16   0.0093        body_rel   0.0086        body_rel   0.0074
9          rsi_14   0.0091      ret_1_lag3   0.0082        vol_z_96   0.0073

Look for features that make sense (e.g. `btc_ret_1` for SOL: BTC tends to move first). A model that leans on
features with no plausible story is a warning sign for overfitting.

## 6. Save, register and activate

Each model is saved as an artifact under `models/direction/<symbol>/<version>.joblib` (not committed to git, see
`.gitignore`: everyone trains locally against their own database) and gets a row in `model_registry` with its
algorithm, features, parameters, metrics and training period.

**Active** means: this is the model `predict()` uses. Only one model per task and symbol can be active. The per-symbol
models are activated here; the shared model is registered for comparison but stays inactive. Set `ACTIVATE_SHARED = True`
to also make it the fallback for symbols without their own model.

In [10]:
ACTIVATE_PER_SYMBOL = True
ACTIVATE_SHARED = False

saved = {}
for t, b in built.items():
    activate = ACTIVATE_SHARED if t is None else ACTIVATE_PER_SYMBOL
    saved[t] = tm.persist(conn, b, activate_model=activate)

pd.DataFrame([tm.summary(b, saved[t]) for t, b in built.items()]).set_index("symbol")[
    ["model", "model_id", "model_version", "test_auc", "active", "artifact_path"]]

,model,model_id,model_version,test_auc,active,artifact_path
symbol,,,,,,
BTCUSDT,hgb,1,20261117T120840Z-42fa5de,0.5100,True,models/direction/BTCUSDT/20261117T120840Z-42fa...
SOLUSDT,hgb,2,20261117T120840Z-42fa5de,0.5103,True,models/direction/SOLUSDT/20261117T120840Z-42fa...
ALL,hgb,3,20261117T120840Z-42fa5de,0.5169,False,models/direction/ALL/20261117T120840Z-42fa5de....


In [11]:
# the newest rows of the model registry
with conn.cursor() as cur:
    cur.execute("""SELECT r.model_id, COALESCE(s.symbol, 'ALL') AS symbol, r.model_version, r.algo, r.is_active,
                          (r.metrics->>'test_auc')::float AS test_auc, r.trained_at
                   FROM model_registry r LEFT JOIN dim_symbol s USING (symbol_id)
                   WHERE r.task = 'direction' ORDER BY r.model_id DESC LIMIT 10""")
    registry = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
registry

,model_id,symbol,model_version,algo,is_active,test_auc,trained_at
0,3,ALL,20261117T120840Z-42fa5de,sklearn.HistGradientBoostingClassifier,False,0.5169,2026-11-17 12:08:40.900034+00:00
1,2,SOLUSDT,20261117T120840Z-42fa5de,sklearn.HistGradientBoostingClassifier,True,0.5103,2026-11-17 12:08:40.853006+00:00
2,1,BTCUSDT,20261117T120840Z-42fa5de,sklearn.HistGradientBoostingClassifier,True,0.5100,2026-11-17 12:08:36.414714+00:00


## 7. Try it: predictions for the API

The API never loads a model: `GET /predict/{symbol}` reads the newest row of the active model from the
`predictions` table. `predict_all()` writes that row for every symbol with an active model: it takes the newest
candle in `gold_features`, asks the model and stores P(up). Same as `python src/models/predict_model.py --all`.

In [12]:
from predict_model import predict_all  # noqa: E402

results, errors = predict_all()
pd.DataFrame(results)

,symbol,open_time,prob_up,predicted_up,model_version
0,BTCUSDT,2026-11-17T11:45:00+00:00,0.4589,False,20261117T120840Z-42fa5de
1,SOLUSDT,2026-11-17T11:45:00+00:00,0.4188,False,20261117T120840Z-42fa5de


In [13]:
with conn.cursor() as cur:
    cur.execute("SELECT symbol, task, open_time, y_pred_value AS prob_up, model_version, predicted_at "
                "FROM v_latest_predictions WHERE task = 'direction'")
    latest = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
conn.close()
latest

,symbol,task,open_time,prob_up,model_version,predicted_at
0,BTCUSDT,direction,2026-11-17 11:45:00+00:00,0.4589,20261117T120840Z-42fa5de,2026-11-17 12:08:40.931431+00:00
1,SOLUSDT,direction,2026-11-17 11:45:00+00:00,0.4188,20261117T120840Z-42fa5de,2026-11-17 12:08:40.967406+00:00
